In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import MinMaxScaler

import torch
from torch import nn, optim
from torch.utils.data import TensorDataset , DataLoader

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
def make_DataSet(x_train, x_val, y_train, y_val, batch_size= 32):
    x_train_tensor = torch.tensor(x_train, dtype= torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype= torch.float32).view(-1,1)
    x_val_tensor = torch.tensor(x_val, dtype= torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype= torch.float32).view(-1,1)

    train_dataset = TensorDataset(x_train_tensor, y_train_tensor)

    train_loader = DataLoader(train_dataset, batch_size= batch_size, shuffle= True)

    return train_loader, x_val_tensor, y_val_tensor

In [4]:
def train(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    tr_loss = 0

    model.train()
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        pred = model(x)
        loss = loss_fn(pred, y)
        tr_loss += loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    tr_loss /= num_batches
    return tr_loss.item()

In [16]:
def evaluate(x_val_tensor, y_val_tensor, model, loss_fn, device):
    model.eval()
    with torch.no_grad():
        x, y = x.to(device), y.to(device)
        pred = model(x)
        eval_loss = loss_fn(pred, y).item()
    return eval_loss, pred

In [6]:
def learning_curve(tr_loss_list, val_loss_list):
    epochs = list(range(1, len(tr_loss_list)+1))
    plt.plot(epochs, tr_loss_list, label='train_err', marker='.')
    plt.plot(epochs, val_loss_list, label='val_err', marker='.')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

In [7]:
import requests
from io import BytesIO
import certifi

In [8]:
path = "https://bit.ly/titanic_simple_csv"
response = requests.get(path, verify= certifi.where(), timeout=15)
response.raise_for_status()
data = pd.read_csv(BytesIO(response.content))
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,7.2500,Southampton
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,71.2833,Cherbourg
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,7.9250,Southampton
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,53.1000,Southampton
4,5,0,3,"Allen, Mr. William Henry",male,35.0,8.0500,Southampton


In [9]:
target = 'Survived'
feature = ['Sex', 'Age', 'Fare']
x = data.loc[:, feature]
y = data.loc[:, target]

In [10]:
# 가변수화
# 성별(Sex)는 범주형 데이터이기 때문에 특징으로 사용하려면 가변수화 필요함

x = pd.get_dummies(x, columns= ['Sex'], drop_first= True)
x.head()

,Age,Fare,Sex_male
0,22.0,7.2500,True
1,38.0,71.2833,False
2,26.0,7.9250,False
3,35.0,53.1000,False
4,35.0,8.0500,True


In [11]:
# 데이터 분할 후 데이터로더 생성
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=.3)

scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

y_train = y_train.values
y_val = y_val.values

train_loader, x_val_tensor, y_val_tensor = make_DataSet(x_train, x_val, y_train, y_val, 32)

In [12]:
# 모델 설계
LR = 1e-2
EPOCH = 50

n_feature = x.shape[1]
model = nn.Sequential(nn.Linear(n_feature, 1),
                      nn.Sigmoid()
        ).to(device)


In [14]:
loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr= LR)

In [ ]:
tr_loss_list, val_loss_list = [], []

for ep in range(EPOCH):
    tr_loss = train(train_loader, model, loss_fn, optimizer, device)
    val_loss = evaluate(x_val_tensor, y_val_tensor, model, loss_fn, device)